# 01 — Preparação do Ambiente Experimental

Este notebook documenta a preparação do ambiente utilizado no experimento de avaliação de defesas contra prompt injection em LLMs.

O objetivo desta etapa é registrar:

- estrutura do projeto;
- ambiente Python;
- dependências utilizadas;
- versões das bibliotecas;
- diretórios utilizados para benchmarks externos, dados, scripts, notebooks, logs, execuções versionadas e exportações.

## 1. Ambiente de execução: RunPod

O experimento foi executado em um ambiente de GPU em nuvem utilizando o RunPod. A escolha do RunPod foi motivada pela necessidade de acessar uma GPU dedicada para treinamento e avaliação de adaptadores LoRA em um modelo Llama 3/3.1 8B Instruct, sem depender de recursos locais.

O uso de uma GPU em nuvem foi importante porque os cenários C2, C3 e C4 envolvem treinamento ou ajuste do modelo, o que exige maior capacidade computacional do que a disponível em uma máquina comum. Além disso, o RunPod permite selecionar diferentes tipos de GPU, configurar disco, volume persistente, portas expostas e acessar o ambiente por terminal, JupyterLab ou outras formas de conexão remota.

De forma geral, o processo de criação do Pod seguiu as seguintes etapas:

1. Acessar a plataforma RunPod.
2. Entrar na área de Pods.
3. Selecionar a opção de Deploy ou criação de novo Pod.
4. Escolher uma GPU compatível com o experimento.
5. Escolher um template com suporte a PyTorch/CUDA.
6. Configurar o volume e o disco do container.
7. Expor portas necessárias, como JupyterLab ou SSH.
8. Iniciar o Pod.
9. Acessar o ambiente por terminal, SSH ou JupyterLab.

O Pod utilizado nesta execução foi configurado com uma GPU dedicada. As informações exatas da GPU são registradas automaticamente mais adiante por meio do comando `nvidia-smi`. No entanto, as demais informações utilizadas estão na tabela a seguir.

| Pod Template        | GPU Count | Container disk | Volume disk |
|---------------------|-----------|----------------|-------------|
| Runpod Pytorch 2.8. | 1         | 50 GB          | 100GB       |

Além disso, a configuração permitiu acesso ao terminal SSH e Jupyter Notebook.

---

Neste experimento, o diretório principal foi organizado em:

```bash
/workspace/pi-defense-exp
```

A pasta `/workspace` foi utilizada porque, em ambientes RunPod, ela é normalmente associada ao armazenamento persistente do Pod. Isso reduz o risco de perda de dados ao reiniciar o ambiente, desde que os arquivos importantes sejam mantidos dentro desse diretório.

Durante o uso do RunPod, também é importante observar alguns cuidados práticos:

* manter dados, scripts, resultados e adaptadores dentro de `/workspace`;
* salvar snapshots ou backups após etapas importantes;
* registrar versões das bibliotecas e informações da GPU;
* usar logs para documentar treinamentos e avaliações;
* evitar armazenar artefatos importantes apenas em diretórios temporários do container;
* baixar os resultados finais para a máquina local ao final do experimento.

## 2. Bootstrap do ambiente e seleção do kernel

Antes de executar as células principais deste notebook, é necessário garantir que o ambiente virtual do projeto foi criado e registrado como kernel do Jupyter.

Este experimento será executado no RunPod, usando a seguinte raiz de projeto:

```text
/workspace/pi-defense-exp
```

O ambiente virtual esperado é:

```text
/workspace/pi-defense-exp/.venv
```

E o kernel Jupyter esperado é:

```text
Python (pi-defense-exp)
```

Essa etapa é importante porque o notebook precisa ser executado com o Python correto. Se outro kernel estiver selecionado, as células podem usar dependências de outro ambiente, causando erros de importação ou inconsistências entre notebooks.

Além disso, no RunPod, a imagem base normalmente já possui `torch` instalado com suporte à versão correta de CUDA. Por isso, o ambiente virtual será criado com `--system-site-packages`, permitindo que o `.venv` enxergue bibliotecas já disponíveis no sistema, especialmente PyTorch/CUDA, sem precisar reinstalá-las manualmente.

O fluxo correto é:

```text
1. Criar a pasta raiz do projeto em /workspace/pi-defense-exp.
2. Criar o ambiente virtual .venv.
3. Atualizar pip, setuptools e wheel.
4. Instalar as dependências leves para dados.
5. Registrar o kernel Jupyter da .venv.
6. Trocar manualmente o kernel do notebook para Python (pi-defense-exp).
7. Executar as células de validação do ambiente.
```

Depois de registrar o kernel, selecione manualmente no Jupyter:

```text
Kernel → Change Kernel → Python (pi-defense-exp)
```

As próximas células farão uma validação explícita do interpretador Python em uso. O caminho esperado será:

```text
/workspace/pi-defense-exp/.venv/bin/python
```

Se o notebook estiver usando outro Python, a execução será interrompida para evitar que arquivos, dependências, logs ou manifestos sejam produzidos no ambiente errado.

### Bootstrap pelo terminal do RunPod

Execute os comandos abaixo no terminal do RunPod antes de continuar o notebook:

```bash
cd /workspace

mkdir -p pi-defense-exp
cd pi-defense-exp

python3 -m venv .venv --system-site-packages
source .venv/bin/activate

python -m pip install --upgrade pip setuptools wheel

python -m pip install \
  datasets \
  pandas \
  numpy \
  scikit-learn \
  scipy \
  statsmodels \
  matplotlib \
  tqdm \
  rich \
  pyyaml \
  ipykernel \
  jupyter \
  jupyterlab

python -m ipykernel install \
  --user \
  --name pi-defense-exp \
  --display-name "Python (pi-defense-exp)"
```

Após executar esses comandos, volte para o Jupyter e selecione o kernel:

```text
Python (pi-defense-exp)
```

Somente depois disso continue a execução das células Python deste notebook.

In [1]:
from pathlib import Path
import os
import sys
import platform

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
os.chdir(PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")
print(f"Python atual: {sys.executable}")
print(f"Sistema: {platform.platform()}")

Project root: /workspace/pi-defense-exp
Python atual: /usr/local/bin/python
Sistema: Linux-6.8.0-40-generic-x86_64-with-glibc2.39


## 3. Criar estrutura de pastas

Nesta etapa, será criada a estrutura principal de diretórios do projeto. A organização das pastas foi definida para manter o experimento reprodutível, separando claramente dados brutos, dados processados, views de treinamento, resultados, logs e manifestos.

A estrutura foi pensada para acompanhar todo o fluxo experimental:

1. preparação do ambiente;
2. criação da base canônica;
3. geração dos ataques;
4. criação das views para StruQ-like SFT, SecAlign-like DPO e Instruction-Hierarchy-like SFT;
5. treinamento dos adaptadores;
6. avaliação dos cenários C0, C1, C2, C3 e C4;
7. cálculo das métricas;
8. armazenamento dos manifestos de reprodutibilidade.

A pasta `data/` concentra os dados usados e gerados pelo experimento. Dentro dela, `raw/` é reservada para dados baixados diretamente das fontes originais, enquanto `canonical/` armazena os arquivos normalizados em um formato comum. A pasta `views/` contém as transformações específicas para cada defesa avaliada.

A pasta `results/` armazena as saídas do experimento, incluindo gerações dos modelos, métricas calculadas e tabelas finais. A pasta `logs/` guarda registros de execução dos notebooks, scripts, treinamento e avaliação. Já a pasta `manifests/` armazena manifestos em `.json` e `.md`, contendo informações de reprodutibilidade, como caminhos gerados, versões de dependências, contagens de exemplos, configurações e decisões experimentais.

A estrutura esperada é:

```text
/workspace/pi-defense-exp/
  configs/
    experiment.yaml

  data/
    raw/
    cache/
    canonical/
    views/
      struq/
      secalign/
      ih/
      evaluation/

  notebooks/

  scripts/

  adapters/
    struq/
    secalign/
    ih/

  results/
    generations/
    metrics/
    tables/
    reports/

  logs/
    environment/
    notebooks/
    data/
    training/
    evaluation/

  manifests/
    environment/
    notebooks/
    data/
    training/
    evaluation/
```

Essa separação evita misturar dados originais, dados derivados, resultados e registros de execução. Também facilita a inspeção posterior dos artefatos produzidos por cada notebook.


In [2]:
from pathlib import Path

PROJECT_ROOT = Path("/workspace/pi-defense-exp")

directories = {
    "configs": "Arquivos de configuração do experimento.",

    "exports": "Arquivos gerados de exportações no formato tar.gz",
    
    "data/raw": "Dados originais baixados das fontes externas.",
    "data/cache": "Arquivos temporários ou cache de datasets.",
    "data/canonical": "Dados normalizados em formato canônico comum.",
    "data/views/struq": "Views para StruQ format-only e StruQ-like SFT.",
    "data/views/secalign": "Views para SecAlign-like DPO.",
    "data/views/ih": "Views para Instruction-Hierarchy-like SFT.",
    "data/views/evaluation": "Arquivos usados na avaliação comum dos cenários.",
    
    "notebooks": "Notebooks do experimento.",
    "scripts": "Scripts auxiliares equivalentes às etapas dos notebooks.",
    
    "adapters/struq": "Adaptadores treinados para o cenário StruQ-like SFT.",
    "adapters/secalign": "Adaptadores treinados para o cenário SecAlign-like DPO.",
    "adapters/ih": "Adaptadores treinados para o cenário Instruction-Hierarchy-like SFT.",
    
    "results/generations": "Saídas brutas geradas pelos modelos durante a avaliação.",
    "results/metrics": "Métricas calculadas a partir das gerações.",
    "results/tables": "Tabelas finais consolidadas.",
    "results/reports": "Relatórios auxiliares e análises intermediárias.",
    
    "logs/environment": "Logs da configuração inicial do ambiente, kernel, GPU e dependências.", 
    "logs/notebooks": "Logs relacionados à execução dos notebooks.",
    "logs/data": "Logs da preparação de dados e geração de ataques.",
    "logs/training": "Logs dos treinamentos SFT e DPO.",
    
    "logs/evaluation": "Logs da avaliação dos cenários experimentais.",
 
    "manifests/environment": "Manifestos do notebook de configuração do ambiente.",
    "manifests/notebooks": "Manifestos dos notebooks executados.",
    "manifests/data": "Manifestos da criação dos datasets e views.",
    "manifests/training": "Manifestos dos treinamentos realizados.",
    "manifests/evaluation": "Manifestos das avaliações e métricas.",
}

for relative_path in directories:
    path = PROJECT_ROOT / relative_path
    path.mkdir(parents=True, exist_ok=True)

print(f"Diretório raiz do projeto: {PROJECT_ROOT}")
print("\nEstrutura de pastas criada/verificada com sucesso:\n")

for relative_path, description in directories.items():
    print(f"- {PROJECT_ROOT / relative_path}")
    print(f"  {description}")

Diretório raiz do projeto: /workspace/pi-defense-exp

Estrutura de pastas criada/verificada com sucesso:

- /workspace/pi-defense-exp/configs
  Arquivos de configuração do experimento.
- /workspace/pi-defense-exp/exports
  Arquivos gerados de exportações no formato tar.gz
- /workspace/pi-defense-exp/data/raw
  Dados originais baixados das fontes externas.
- /workspace/pi-defense-exp/data/cache
  Arquivos temporários ou cache de datasets.
- /workspace/pi-defense-exp/data/canonical
  Dados normalizados em formato canônico comum.
- /workspace/pi-defense-exp/data/views/struq
  Views para StruQ format-only e StruQ-like SFT.
- /workspace/pi-defense-exp/data/views/secalign
  Views para SecAlign-like DPO.
- /workspace/pi-defense-exp/data/views/ih
  Views para Instruction-Hierarchy-like SFT.
- /workspace/pi-defense-exp/data/views/evaluation
  Arquivos usados na avaliação comum dos cenários.
- /workspace/pi-defense-exp/notebooks
  Notebooks do experimento.
- /workspace/pi-defense-exp/scripts
  S

## 4. Criar arquivos de dependências

Nesta etapa, serão criados os arquivos de dependências do projeto. Esses arquivos documentam quais bibliotecas Python são necessárias para executar as diferentes fases do experimento e ajudam a tornar o ambiente mais reprodutível.

As dependências foram separadas em dois grupos principais:

```text
requirements-data.txt
requirements-train.txt
```

O arquivo `requirements-data.txt` contém as bibliotecas necessárias para as etapas iniciais do experimento, como carregamento de datasets, manipulação de dados, geração dos arquivos canônicos, criação dos ataques e construção das views de treinamento.

Já o arquivo `requirements-train.txt` contém bibliotecas relacionadas ao treinamento e ajuste dos modelos, incluindo ferramentas do ecossistema Hugging Face para fine-tuning, uso de adaptadores e otimização por preferência.

Essa separação é importante porque nem todas as etapas do projeto exigem as dependências mais pesadas de treinamento. Assim, é possível validar primeiro a preparação dos dados antes de avançar para as fases computacionalmente mais custosas.

No ambiente RunPod, o `torch` normalmente já vem instalado na imagem base com suporte à versão correta de CUDA. Por isso, ele não será fixado diretamente nos arquivos de dependências. Essa decisão evita conflitos entre a versão do PyTorch instalada pela imagem do RunPod e uma versão instalada posteriormente via `pip`.

A organização esperada é:

```text
requirements-data.txt
  Dependências para preparação, inspeção e validação dos dados.

requirements-train.txt
  Dependências para fine-tuning, DPO, adaptadores e avaliação com modelos.
```

Essa etapa não instala ainda todas as dependências obrigatoriamente. O objetivo principal aqui é registrar os arquivos que descrevem o ambiente necessário para a execução do experimento.


In [3]:
requirements_data = """
ipykernel
jupyter
notebook
jupyterlab
pandas
numpy
scikit-learn
datasets
huggingface_hub
pyarrow
tqdm
matplotlib
""".strip()

requirements_train = """
torch
transformers
accelerate
peft
trl
bitsandbytes
sentencepiece
protobuf
""".strip()

Path("requirements-data.txt").write_text(
    requirements_data + "\n",
    encoding="utf-8"
)

Path("requirements-train.txt").write_text(
    requirements_train + "\n",
    encoding="utf-8"
)

print("Criados: requirements-data.txt e requirements-train.txt")

Criados: requirements-data.txt e requirements-train.txt


## 5. Criar `.venv`

Nesta etapa, será criado o ambiente virtual Python utilizado pelo projeto. O objetivo é isolar as dependências do experimento em um ambiente próprio, evitando conflitos com bibliotecas instaladas globalmente no sistema ou em outros projetos.

O ambiente virtual será criado dentro da raiz do projeto, no diretório:

```text id="ggug6h"
/workspace/pi-defense-exp/.venv
```

Neste experimento, o nome `.venv` será utilizado para deixar explícito que o ambiente está associado à avaliação de defesas contra prompt injection baseada em dados derivados de Open Prompt Injection e tarefas relacionadas.

Como o experimento será executado no RunPod, a criação do ambiente virtual deve considerar que a imagem base pode já conter bibliotecas importantes instaladas, especialmente `torch` com suporte a CUDA. Por esse motivo, o ambiente será criado com acesso aos pacotes do sistema usando a opção `--system-site-packages`.

Essa decisão é importante porque evita reinstalar o PyTorch manualmente e reduz o risco de incompatibilidade entre a versão do `torch`, a versão do CUDA e os drivers disponíveis na GPU do ambiente.

A criação do `.venv` permite:

```text id="qnbxks"
- manter as dependências do experimento organizadas;
- registrar um kernel específico para os notebooks;
- evitar misturar pacotes de projetos diferentes;
- facilitar a reexecução do experimento no mesmo ambiente;
- preservar o PyTorch/CUDA já disponível na imagem do RunPod.
```

Após a criação do ambiente virtual, ele será usado como base para registrar o kernel Jupyter:

```text id="6y9jz3"
Python (pi-defense-exp)
```

Esse kernel deverá ser selecionado manualmente nos notebooks seguintes antes da execução das etapas de preparação de dados, treinamento e avaliação.


In [4]:
import subprocess

VENV_DIR = PROJECT_ROOT / ".venv"

if not VENV_DIR.exists():
    print("Criando .venv...")
    subprocess.run(
        [sys.executable, "-m", "venv", "--system-site-packages", str(VENV_DIR)],
        check=True,
    )
else:
    print(".venv já existe.")

if platform.system().lower().startswith("win"):
    VENV_PYTHON = VENV_DIR / "Scripts" / "python.exe"
else:
    VENV_PYTHON = VENV_DIR / "bin" / "python"

print(f"Python do .venv: {VENV_PYTHON}")
print(f"Existe? {VENV_PYTHON.exists()}")

Criando .venv...
Python do .venv: /workspace/pi-defense-exp/.venv/bin/python
Existe? True


## 6. Instalar dependências leves

Nesta etapa, serão instaladas apenas as dependências necessárias para as fases iniciais do experimento. O foco aqui é preparar o ambiente para baixar datasets, manipular arquivos, normalizar exemplos, gerar ataques e validar a consistência dos dados produzidos.

As dependências leves ficam registradas no arquivo:

```text
requirements-data.txt
```

Esse arquivo deve conter bibliotecas voltadas à preparação e análise dos dados, como `datasets`, `pandas`, `numpy`, `scikit-learn`, `pyyaml`, `tqdm` e `ipykernel`.

Nesta etapa, ainda não serão instaladas dependências mais pesadas relacionadas ao treinamento de modelos, como `transformers`, `accelerate`, `peft` e `trl`, caso elas não sejam necessárias para a preparação dos dados. Essas bibliotecas serão tratadas separadamente nas etapas de treinamento.

Essa separação é importante porque permite validar primeiro se a base canônica, os ataques e as views experimentais estão sendo gerados corretamente antes de avançar para a parte mais custosa do projeto.

No contexto deste experimento, esta célula deve garantir que o ambiente consiga executar as seguintes tarefas:

```text
- carregar datasets do Hugging Face;
- manipular tabelas e arquivos JSONL;
- criar diretórios e arquivos de configuração;
- gerar splits de treino, validação e teste;
- construir exemplos limpos e atacados;
- validar contagens e consistência dos dados;
- registrar o kernel Jupyter associado ao ambiente virtual.
```

Como o experimento será executado em um ambiente RunPod com GPU, bibliotecas como `torch` não serão instaladas nesta etapa. O PyTorch normalmente já vem instalado na imagem base com suporte à versão correta de CUDA, e reinstalá-lo manualmente pode causar conflitos com o ambiente da GPU.

Assim, esta célula instala apenas o necessário para a preparação dos dados, mantendo o ambiente inicial mais simples, estável e fácil de depurar.


In [5]:
subprocess.run([str(VENV_PYTHON), "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"], check=True)
subprocess.run([str(VENV_PYTHON), "-m", "pip", "install", "-r", "requirements-data.txt"], check=True)

print("Dependências leves instaladas.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 272.4 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 177.1 kB/s eta 0:00:0000:0100:01
  Attempting uninstall: setuptools
    Found existing installation: setuptools 80.9.0
    Not uninstalling setuptools at /usr/local/lib/python3.12/dist-packages, outside environment /workspace/pi-defense-exp/.venv
    Can't uninstall 'setuptools'. No files were found to uninstall.
  Attempting uninstall: pip
    Found existing installation: pip 24.0
    Uninstalling pip-24.0:
      Successfully uninstalled pip-24.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 282.2 kB/s  0:00:38 eta 0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 289.7 kB/s  0:00:31m0:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 139.3 kB/s  0:00:02m-:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.1/721.1 kB 335.4 kB/s  0:00:02eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5

## 7. Instalar kernel Jupyter do `.venv`

Nesta etapa, o ambiente virtual criado anteriormente será registrado como um kernel do Jupyter. Isso permite que os notebooks sejam executados diretamente com o Python localizado dentro do `.venv`, garantindo que as dependências instaladas sejam as mesmas usadas durante a execução do experimento.

O kernel será registrado com o nome:

```text
Python (pi-defense-exp)
```

Após executar a célula abaixo, será necessário trocar manualmente o kernel do notebook no Jupyter para:

```text
Python (pi-defense-exp)
```

Essa troca é importante porque a criação do `.venv` por si só não altera automaticamente o interpretador usado pelo notebook atual. Se o kernel antigo continuar selecionado, o notebook poderá usar outro Python, com dependências diferentes ou incompletas.

Depois de trocar o kernel, as próximas células farão uma validação explícita do interpretador Python em uso. O caminho esperado será:

```text
/workspace/pi-defense-exp/.venv/bin/python
```

Caso o notebook esteja usando outro interpretador, a execução será interrompida para evitar que etapas importantes sejam feitas no ambiente errado.

Essa validação ajuda a garantir que os dados, manifestos, logs e demais artefatos sejam produzidos de forma consistente e reprodutível ao longo dos notebooks.

In [6]:
import subprocess

KERNEL_NAME = "pi-defense-exp"
DISPLAY_NAME = "Python (pi-defense-exp)"

subprocess.run(
    [
        str(VENV_PYTHON),
        "-m",
        "ipykernel",
        "install",
        "--user",
        "--name",
        KERNEL_NAME,
        "--display-name",
        DISPLAY_NAME,
    ],
    check=True,
)

print(f"Kernel instalado: {DISPLAY_NAME}")
print("Agora troque manualmente o kernel e reinicie o notebook.")

Installed kernelspec pi-defense-exp in /root/.local/share/jupyter/kernels/pi-defense-exp
Kernel instalado: Python (pi-defense-exp)
Agora troque manualmente o kernel e reinicie o notebook.


## 8. Verificar kernel atual

Este notebook deve ser executado com o kernel principal do projeto:

```text
Python (pi-defense-exp)
```

Esse kernel deve apontar para:

```bash
/workspace/pi-defense-exp/.venv/bin/python
```

In [1]:
import sys
from pathlib import Path

EXPECTED_PYTHON = Path("/workspace/pi-defense-exp/.venv/bin/python")
CURRENT_PYTHON = Path(sys.executable)

print("Python atual:", CURRENT_PYTHON)
print("Python esperado para este notebook:", EXPECTED_PYTHON)

if CURRENT_PYTHON != EXPECTED_PYTHON:
    raise RuntimeError(
        "Este notebook deve ser executado com o kernel 'Python (pi-defense-exp)'. "
        "Selecione esse kernel no Jupyter antes de continuar."
    )

print("Validação inicial concluída.")
print("Kernel correto para o notebook 01.")

Python atual: /workspace/pi-defense-exp/.venv/bin/python
Python esperado para este notebook: /workspace/pi-defense-exp/.venv/bin/python
Validação inicial concluída.
Kernel correto para o notebook 01.


## 9. Checar pacotes principais

Nesta etapa, serão verificadas as principais bibliotecas Python necessárias para a continuidade do experimento. A checagem não tem como objetivo apenas listar pacotes instalados, mas confirmar se eles estão disponíveis para importação no kernel atualmente selecionado.

Essa validação é importante porque, em ambientes com múltiplos interpretadores Python, como notebooks Jupyter executando dentro de um `.venv`, pode acontecer de uma biblioteca estar instalada em um ambiente, mas não estar acessível no kernel em uso.

A checagem considera dois aspectos:

```text
- se a distribuição do pacote está instalada;
- se o módulo Python correspondente pode ser encontrado para import.
```

Entre os pacotes verificados estão bibliotecas essenciais para as próximas etapas do projeto, como `datasets`, `pandas`, `numpy`, `transformers`, `accelerate`, `peft`, `trl` e `huggingface_hub`.

Embora nem todos esses pacotes sejam usados imediatamente no notebook de configuração, eles fazem parte do fluxo completo do experimento, que envolve preparação de dados, fine-tuning, otimização por preferência, uso de adaptadores e avaliação dos modelos.

Caso algum pacote obrigatório não esteja disponível para importação, a execução será interrompida. Isso evita avançar para etapas posteriores com um ambiente incompleto ou inconsistente.

In [2]:
from importlib.metadata import version, PackageNotFoundError
import importlib.util
import pandas as pd

packages_to_check = [
    ("pandas", "pandas"),
    ("numpy", "numpy"),
    ("sklearn", "scikit-learn"),
    ("datasets", "datasets"),
    ("huggingface_hub", "huggingface-hub"),
    ("pyarrow", "pyarrow"),
    ("tqdm", "tqdm"),
    ("matplotlib", "matplotlib"),
]

package_rows = []

for import_name, dist_name in packages_to_check:
    try:
        pkg_version = version(dist_name)
        installed = True
    except PackageNotFoundError:
        pkg_version = None
        installed = False

    import_available = importlib.util.find_spec(import_name) is not None

    if installed and import_available:
        status = "ok"
    elif installed and not import_available:
        status = "installed_but_import_not_found"
    elif not installed and import_available:
        status = "import_found_but_distribution_not_found"
    else:
        status = "missing"

    package_rows.append({
        "package": import_name,
        "distribution": dist_name,
        "version": pkg_version,
        "installed": installed,
        "import_available": import_available,
        "status": status,
    })

package_check_df = pd.DataFrame(package_rows)
display(package_check_df)

if not package_check_df["import_available"].all():
    missing_imports = package_check_df.loc[
        ~package_check_df["import_available"],
        "package",
    ].tolist()

    raise RuntimeError(
        "Algumas dependências obrigatórias não estão disponíveis para import: "
        + ", ".join(missing_imports)
    )

,package,distribution,version,installed,import_available,status
0,pandas,pandas,3.0.4,True,True,ok
1,numpy,numpy,2.1.2,True,True,ok
2,sklearn,scikit-learn,1.9.0,True,True,ok
3,datasets,datasets,5.0.0,True,True,ok
4,huggingface_hub,huggingface-hub,1.21.0,True,True,ok
5,pyarrow,pyarrow,24.0.0,True,True,ok
6,tqdm,tqdm,4.68.3,True,True,ok
7,matplotlib,matplotlib,3.11.0,True,True,ok


## 10. Criar configuração inicial

Nesta etapa, será criado o arquivo inicial de configuração do experimento. Esse arquivo centraliza parâmetros importantes que serão reutilizados pelos notebooks seguintes, evitando que decisões como caminhos, nomes de tarefas, tamanhos de split, tipos de ataque e seed experimental fiquem espalhadas em várias células.

A configuração será salva em:

```text
configs/experiment.yaml
```

Esse arquivo funciona como uma referência comum para manter a consistência entre as etapas do projeto. Em vez de redefinir manualmente os mesmos valores em cada notebook, os próximos notebooks poderão carregar essa configuração e reutilizar os parâmetros definidos aqui.

A configuração inicial deve incluir informações como:

```text
- diretório raiz do projeto;
- seed experimental;
- nomes das tarefas avaliadas;
- tamanhos dos splits de treino, validação e teste;
- ataques vistos durante o treinamento;
- ataques não vistos ou adaptativos usados apenas na avaliação;
- caminhos esperados para dados, views, resultados, logs e manifestos;
- nomes dos cenários experimentais C0, C1, C2, C3 e C4.
```

Essa centralização ajuda a tornar o experimento mais reprodutível, porque documenta as principais decisões experimentais em um único arquivo versionável e legível.

Além disso, o arquivo `experiment.yaml` será usado como base para os manifestos finais dos notebooks, permitindo registrar quais configurações estavam ativas no momento em que cada etapa foi executada.


In [4]:
experiment_yaml = """
seed: 42
project_name: pi-defense-eval

splits:
  train_per_task: 300
  validation_per_task: 50
  test_per_task: 268

tasks:
  - mrpc
  - rte
  - cola
  - qqp
  - sst2
  - sms_spam
  - hsol

seen_attacks:
  - naive
  - ignore
  - escape
  - fake_comp
  - combine

unseen_attacks:
  - combine_adaptive
  - gcg
  - gcg_adaptive

scenarios:
  - c0_base
  - c1_struq_format_only
  - c2_struq_sft
  - c3_secalign_dpo
  - c4_ih_sft
""".strip()

Path("../configs/experiment.yaml").write_text(experiment_yaml + "\n", encoding="utf-8")
print(Path("../configs/experiment.yaml").read_text(encoding="utf-8"))

seed: 42
project_name: pi-defense-eval

splits:
  train_per_task: 300
  validation_per_task: 50
  test_per_task: 268

tasks:
  - mrpc
  - rte
  - cola
  - qqp
  - sst2
  - sms_spam
  - hsol

seen_attacks:
  - naive
  - ignore
  - escape
  - fake_comp
  - combine

unseen_attacks:
  - combine_adaptive
  - gcg
  - gcg_adaptive

scenarios:
  - c0_base
  - c1_struq_format_only
  - c2_struq_sft
  - c3_secalign_dpo
  - c4_ih_sft



## 11. Gerar manifesto do ambiente

Nesta etapa final, será gerado o manifesto do notebook de configuração do ambiente. O manifesto registra as principais informações produzidas e verificadas durante a execução deste notebook, permitindo rastrear posteriormente qual ambiente foi usado na preparação do experimento.

Serão gerados dois arquivos:

```text
manifests/environment/01_environment_setup_manifest.json
manifests/environment/01_environment_setup_manifest.md
```

O arquivo `.json` é mais adequado para leitura automatizada por scripts. Já o arquivo `.md` é mais adequado para inspeção manual, pois apresenta as mesmas informações de forma legível.

O manifesto inclui informações como:

```text
- data e hora de geração;
- diretório raiz do projeto;
- caminho do Python atual;
- caminho do Python esperado;
- nome do kernel esperado;
- versão do Python;
- sistema operacional;
- disponibilidade de GPU;
- saída resumida do nvidia-smi;
- pacotes principais verificados;
- arquivos de dependência criados;
- arquivo de configuração inicial;
- diretórios principais criados.
```

Esse manifesto encerra o notebook de ambiente e serve como evidência de que a configuração inicial foi executada corretamente antes da criação dos datasets.

In [5]:
import json
import platform
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
import importlib.util

PROJECT_ROOT = Path("/workspace/pi-defense-exp")
EXPECTED_PYTHON = Path("/workspace/pi-defense-exp/.venv/bin/python")
CURRENT_PYTHON = Path(sys.executable)
EXPECTED_KERNEL = "Python (pi-defense-exp)"

MANIFEST_DIR = PROJECT_ROOT / "manifests" / "environment"
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

LOG_DIR = PROJECT_ROOT / "logs" / "environment"
LOG_DIR.mkdir(parents=True, exist_ok=True)

manifest_json_path = MANIFEST_DIR / "01_environment_setup_manifest.json"
manifest_md_path = MANIFEST_DIR / "01_environment_setup_manifest.md"

packages_to_check = [
    ("torch", "torch"),
    ("transformers", "transformers"),
    ("accelerate", "accelerate"),
    ("peft", "peft"),
    ("trl", "trl"),
    ("datasets", "datasets"),
    ("huggingface_hub", "huggingface-hub"),
    ("pandas", "pandas"),
    ("numpy", "numpy"),
]

package_rows = []

for import_name, dist_name in packages_to_check:
    try:
        pkg_version = version(dist_name)
        installed = True
    except PackageNotFoundError:
        pkg_version = None
        installed = False

    import_available = importlib.util.find_spec(import_name) is not None

    if installed and import_available:
        status = "ok"
    elif installed and not import_available:
        status = "installed_but_import_not_found"
    elif not installed and import_available:
        status = "import_found_but_distribution_not_found"
    else:
        status = "missing"

    package_rows.append({
        "package": import_name,
        "distribution": dist_name,
        "version": pkg_version,
        "installed": installed,
        "import_available": import_available,
        "status": status,
    })

def run_command(command):
    try:
        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            check=False,
        )
        return {
            "returncode": result.returncode,
            "stdout": result.stdout.strip(),
            "stderr": result.stderr.strip(),
        }
    except FileNotFoundError:
        return {
            "returncode": None,
            "stdout": "",
            "stderr": f"Command not found: {command[0]}",
        }

nvidia_smi_result = run_command(["nvidia-smi"])

required_paths = [
    "configs",
    "data/raw",
    "data/cache",
    "data/canonical",
    "data/views/struq",
    "data/views/secalign",
    "data/views/ih",
    "data/views/evaluation",
    "notebooks",
    "scripts",
    "adapters/struq",
    "adapters/secalign",
    "adapters/ih",
    "results/generations",
    "results/metrics",
    "results/tables",
    "results/reports",
    "logs/environment",
    "logs/notebooks",
    "logs/data",
    "logs/training",
    "logs/evaluation",
    "manifests/environment",
    "manifests/notebooks",
    "manifests/data",
    "manifests/training",
    "manifests/evaluation",
]

path_status = {
    relative_path: (PROJECT_ROOT / relative_path).exists()
    for relative_path in required_paths
}

requirements_files = {
    "requirements-data.txt": (PROJECT_ROOT / "requirements-data.txt").exists(),
    "requirements-train.txt": (PROJECT_ROOT / "requirements-train.txt").exists(),
}

config_files = {
    "configs/experiment.yaml": (PROJECT_ROOT / "configs" / "experiment.yaml").exists(),
}

manifest = {
    "notebook": "01_environment_setup",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(PROJECT_ROOT),
    "expected_kernel": EXPECTED_KERNEL,
    "expected_python": str(EXPECTED_PYTHON),
    "current_python": str(CURRENT_PYTHON),
    "kernel_validation": {
        "matches_expected_python": CURRENT_PYTHON == EXPECTED_PYTHON,
    },
    "python": {
        "version": sys.version,
        "executable": str(CURRENT_PYTHON),
    },
    "system": {
        "platform": platform.platform(),
        "processor": platform.processor(),
        "machine": platform.machine(),
    },
    "gpu": {
        "nvidia_smi_available": nvidia_smi_result["returncode"] == 0,
        "nvidia_smi_returncode": nvidia_smi_result["returncode"],
        "nvidia_smi_stdout": nvidia_smi_result["stdout"],
        "nvidia_smi_stderr": nvidia_smi_result["stderr"],
    },
    "packages": package_rows,
    "requirements_files": requirements_files,
    "config_files": config_files,
    "directories": path_status,
}

with open(manifest_json_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

package_table_md = "\n".join(
    [
        "| Package | Distribution | Version | Import available | Status |",
        "|---|---|---:|---:|---|",
        *[
            (
                f"| {row['package']} "
                f"| {row['distribution']} "
                f"| {row['version']} "
                f"| {row['import_available']} "
                f"| {row['status']} |"
            )
            for row in package_rows
        ],
    ]
)

directory_table_md = "\n".join(
    [
        "| Path | Exists |",
        "|---|---:|",
        *[
            f"| `{relative_path}` | {exists} |"
            for relative_path, exists in path_status.items()
        ],
    ]
)

manifest_md = f"""# Manifesto do ambiente — Notebook 01

## Identificação

- Notebook: `01_environment_setup`
- Gerado em UTC: `{manifest["created_at_utc"]}`
- Diretório raiz do projeto: `{PROJECT_ROOT}`

## Kernel e Python

- Kernel esperado: `{EXPECTED_KERNEL}`
- Python esperado: `{EXPECTED_PYTHON}`
- Python atual: `{CURRENT_PYTHON}`
- Python atual corresponde ao esperado: `{CURRENT_PYTHON == EXPECTED_PYTHON}`

## Sistema

- Plataforma: `{platform.platform()}`
- Máquina: `{platform.machine()}`
- Processador: `{platform.processor()}`

## GPU

- `nvidia-smi` disponível: `{nvidia_smi_result["returncode"] == 0}`
- Código de retorno: `{nvidia_smi_result["returncode"]}`

## Pacotes principais

{package_table_md}

## Arquivos de dependências

- `requirements-data.txt`: `{requirements_files["requirements-data.txt"]}`
- `requirements-train.txt`: `{requirements_files["requirements-train.txt"]}`

## Arquivos de configuração

- `configs/experiment.yaml`: `{config_files["configs/experiment.yaml"]}`

## Diretórios esperados

{directory_table_md}
"""

with open(manifest_md_path, "w", encoding="utf-8") as f:
    f.write(manifest_md)

print("Manifesto JSON criado em:", manifest_json_path)
print("Manifesto Markdown criado em:", manifest_md_path)

Manifesto JSON criado em: /workspace/pi-defense-exp/manifests/environment/01_environment_setup_manifest.json
Manifesto Markdown criado em: /workspace/pi-defense-exp/manifests/environment/01_environment_setup_manifest.md


## 11. Próximo passo

Depois que o kernel correto estiver selecionado, avance para:

```text
02_dataset_creation.ipynb
```

O próximo notebook irá baixar/carregar as tasks, normalizar os dados, criar os ataques e gerar as views de treino.